# Notebook 09: Adaptation on the device without expert labels (E13, hypothesis H3), plus the shift-augmented Phase 1 counterparts (E11b)

**Question.** Starting from a frozen model, can a handful of **unlabelled** local images close the cross-country gap? Three label-free methods and one labelled reference are applied with N ∈ {10, 25, 50, 100, 200, 400} target images drawn from the CCMT `adapt` halves (three draws each) and scored on the CCMT `test` halves, which no adaptation ever sees:

- **AdaBN** (Li et al., 2018): batch-normalisation statistics re-estimated on the N images; no weight changes.
- **Tent** (Wang et al., 2021): after AdaBN, the normalisation layers' scale and shift are trained to minimise prediction entropy.
- **Gated self-training**: the frozen abstention rule (Phase 1 temperature and threshold) picks which images the model may label itself; accepted images fine-tune the final blocks; two rounds.
- **Labelled few-shot reference**: the same N images with their true labels, as the upper reference for what labels would buy.

**Starting models.** (a) The frozen Phase 1 EfficientNet-B0 (seed 42). (b) Its shift-augmented counterpart, trained here first (**E11b**: the Phase 1 recipe and training data with the shift-aimed augmentation of E11, full label set, seed 42; MobileNetV2 likewise, for Notebook 10). E11b models are calibrated on the Phase 1 temperature and threshold partitions with the Phase 1 rules.

**Decision rule (registered).** The label-free method and N with the highest mean macro-F1 on `d3m_test` (Phase 1 maize model) is written to `records/v3_adaptation_registered.json` with a SHA-256 and applied once to the sealed KaraAgro half in Notebook 11. Everything here is **exploratory** (D3 was seen in Phase 1).

**Runtime.** A100. Two sessions: set `CROPS_TO_RUN = ['maize']` in one and `['cassava']` in the other (the E11b training in cell 4 runs once per crop too). Everything is resumable. Cell 7 can be run in either session once both have finished.

In [ ]:
# 1. Setup, integrity checks, environment
CROPS_TO_RUN = ['cassava', 'maize']
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, math, collections
import numpy as np, pandas as pd
import tensorflow as tf, keras
from sklearn.metrics import f1_score, accuracy_score
DRIVE = '/content/drive/MyDrive/ocdf'; WORK = '/content/working'
RUNS = f'{DRIVE}/runs_v3'; OUT = f'{DRIVE}/records/nb09'; RUNDIR = f'{OUT}/e13_runs'
for d in [RUNS, OUT, RUNDIR, WORK]: os.makedirs(d, exist_ok=True)
txt_path = f'{DRIVE}/records/protocol_v3_registration_text.md'
assert hashlib.sha256(open(txt_path, 'rb').read()).hexdigest() == open(txt_path + '.sha256').read().strip(), 'registration text changed'
reg = json.load(open(f'{DRIVE}/records/protocol_v3_registration.json')); reg_sha = open(f'{DRIVE}/records/protocol_v3_registration.json.sha256').read().strip()
fpath = f'{DRIVE}/records/frozen_choices_v2.json'
assert hashlib.sha256(open(fpath, 'rb').read()).hexdigest() == open(f'{DRIVE}/records/frozen_choices_v2.sha256').read().strip(), 'frozen choices changed'
F = json.load(open(fpath)); DEP = F['deployed_instance']; protocol2 = json.load(open(f'{DRIVE}/records/protocol_v2.json'))
for crop, d in DEP.items(): assert d['architecture'] == 'EfficientNetB0' and int(d['seed']) == 42, d
print('registration', reg['registered_utc'], '| Phase 1 deployed:', {c: (round(d['temperature_T'], 4), d['threshold']) for c, d in DEP.items()})
RECIPE = {'head_lr': 1e-3, 'head_max_epochs': 5, 'finetune_lr': 1e-5, 'finetune_max_epochs': 25, 'patience': 5,
          'unfreeze_from': {'MobileNetV2': 'block_13', 'MobileNetV3Small': 'expanded_conv_8', 'EfficientNetB0': 'block6a'}}   # Phase 1 recipe (protocol v2)
SEEDS = [42]; BATCH = 32; IMG = 224
gpus = tf.config.list_physical_devices('GPU'); assert gpus, 'Choose an A100 runtime'
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
env = {'notebook': '09_adaptation', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
json.dump(env, open(f'{OUT}/environment_nb09.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Manifest v3 (integrity), working images, partitions
seal = json.load(open(f'{DRIVE}/records/phase2_data_v3.json'))
mpath = f'{DRIVE}/manifests/{seal["manifest_v3"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == seal['sha256'], 'manifest_v3 changed since it was frozen'
M = pd.read_csv(mpath, dtype=str)
M = M[M.partition != 'd5_test'].copy()                      # SEALED: never loaded in this notebook
for src in ['D1', 'D3', 'D4', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M02', 'M03', 'M04', 'M05']}          # Phase 1 label sets (full)
N_OUT = {c: len(v) for c, v in CLASSES.items()}; LABEL_IDX = {c: {l: i for i, l in enumerate(v)} for c, v in CLASSES.items()}
INV = {c: {i: l for l, i in LABEL_IDX[c].items()} for c in CLASSES}
EVAL_LABELS = {'cassava': [0, 1, 2], 'maize': [0, 4]}                                               # labels CCMT shares with the Phase 1 sets
ADAPT = {'cassava': M[(M.partition == 'd3c_adapt') & (M.research_label.isin(CLASSES['cassava']))].sort_values('sample_id'),
         'maize':   M[(M.partition == 'd3m_adapt') & (M.research_label.isin(['M01', 'M05']))].sort_values('sample_id')}
TESTDF = {'cassava': M[(M.partition == 'd3c_test') & (M.research_label.isin(CLASSES['cassava']))].sort_values('sample_id'),
          'maize':   M[(M.partition == 'd3m_test') & (M.research_label.isin(['M01', 'M05']))].sort_values('sample_id')}
TESTNAME = {'cassava': 'd3c_test', 'maize': 'd3m_test'}
NS = [10, 25, 50, 100, 200, 400]; DRAWS = [1, 2, 3]; METHODS = ['adabn', 'tent', 'self_training', 'labelled']
for c in CLASSES: print(c, 'adapt', len(ADAPT[c]), ADAPT[c].research_label.value_counts().to_dict(), '| test', len(TESTDF[c]), TESTDF[c].research_label.value_counts().to_dict())
DEV = M[M.partition.isin(['train', 'select', 'calib_T', 'thresh'])].copy()                           # Phase 1 development partitions (D1, D4, D6)

In [ ]:
# 3. Data pipeline (same code as Notebook 07) (all augmentation applies to training batches only)
def load_img(path, img):
    x = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    x = tf.image.resize_with_pad(x, img, img, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(x), 0, 255), tf.uint8)
def base_ds(df, classes, img):
    paths = [f'{WORK}/{p}' for p in df.working_path]; y = [classes.index(l) for l in df.research_label]
    return tf.data.Dataset.from_tensor_slices((paths, y)).map(lambda p, l: (load_img(p, img), l), num_parallel_calls=tf.data.AUTOTUNE).cache()

def phase1_aug(seed):
    return keras.Sequential([keras.layers.RandomFlip('horizontal_and_vertical', seed=seed), keras.layers.RandomRotation(0.05, fill_mode='reflect', seed=seed),
                             keras.layers.RandomBrightness(0.1, value_range=(0, 255), seed=seed), keras.layers.RandomContrast(0.1, seed=seed)])

def random_resized_crop(x, img):
    B = tf.shape(x)[0]
    area = tf.random.uniform([B], 0.5, 1.0); ratio = tf.exp(tf.random.uniform([B], tf.math.log(0.75), tf.math.log(4 / 3)))
    w = tf.minimum(tf.sqrt(area * ratio), 1.0); h = tf.minimum(tf.sqrt(area / ratio), 1.0)
    x1 = tf.random.uniform([B]) * (1 - w); y1 = tf.random.uniform([B]) * (1 - h)
    boxes = tf.stack([y1, x1, y1 + h, x1 + w], axis=1)
    return tf.image.crop_and_resize(x, boxes, tf.range(B), [img, img])
def colour_jitter(x):
    B = tf.shape(x)[0]; sh = [B, 1, 1, 1]
    x = x + tf.random.uniform(sh, -0.25, 0.25) * 255.0                                           # brightness
    m = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True); x = m + (x - m) * tf.random.uniform(sh, 0.75, 1.25)   # contrast
    g = tf.image.rgb_to_grayscale(x); x = g + (x - g) * tf.random.uniform(sh, 0.75, 1.25)                     # saturation
    x = tf.clip_by_value(x, 0, 255)
    x = tf.map_fn(lambda im: tf.image.random_hue(im / 255.0, 0.05) * 255.0, x)                               # hue, per image
    return tf.clip_by_value(x, 0, 255)
def random_erasing(x, img, p=0.5, max_area=0.25):
    B = tf.shape(x)[0]
    area = tf.random.uniform([B], 0.02, max_area) * img * img; ratio = tf.exp(tf.random.uniform([B], tf.math.log(0.3), tf.math.log(3.3)))
    h = tf.minimum(tf.cast(tf.sqrt(area * ratio), tf.int32), img); w = tf.minimum(tf.cast(tf.sqrt(area / ratio), tf.int32), img)
    y1 = tf.cast(tf.random.uniform([B]) * tf.cast(img - h, tf.float32), tf.int32); x1 = tf.cast(tf.random.uniform([B]) * tf.cast(img - w, tf.float32), tf.int32)
    ys = tf.range(img)[None, :, None]; xs = tf.range(img)[None, None, :]
    inside = (ys >= y1[:, None, None]) & (ys < (y1 + h)[:, None, None]) & (xs >= x1[:, None, None]) & (xs < (x1 + w)[:, None, None])
    apply = tf.random.uniform([B]) < p
    mask = inside & apply[:, None, None]
    return tf.where(mask[..., None], 127.5, x)
def cutmix(x, y, sw, img, p=0.5):
    B = tf.shape(x)[0]; lam = tf.random.uniform([])
    cut = tf.sqrt(1.0 - lam); ch = tf.cast(cut * img, tf.int32); cw = ch
    cy = tf.random.uniform([], 0, img, dtype=tf.int32); cx = tf.random.uniform([], 0, img, dtype=tf.int32)
    y1 = tf.clip_by_value(cy - ch // 2, 0, img); y2 = tf.clip_by_value(cy + ch // 2, 0, img); x1 = tf.clip_by_value(cx - cw // 2, 0, img); x2 = tf.clip_by_value(cx + cw // 2, 0, img)
    ys = tf.range(img)[:, None]; xs = tf.range(img)[None, :]
    box = ((ys >= y1) & (ys < y2) & (xs >= x1) & (xs < x2))[None, :, :, None]
    xr, yr, sr = tf.reverse(x, [0]), tf.reverse(y, [0]), tf.reverse(sw, [0])
    lam_adj = 1.0 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / float(img * img)
    xm = tf.where(box, xr, x); ym = lam_adj * y + (1 - lam_adj) * yr; sm = lam_adj * sw + (1 - lam_adj) * sr
    do = tf.random.uniform([]) < p
    return tf.cond(do, lambda: (xm, ym, sm), lambda: (x, y, sw))

def train_ds(base, n_classes, seed, weights, img, aug_kind):
    w = tf.constant(weights, tf.float32); p1 = phase1_aug(seed)
    ds = base.shuffle(4096, seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes), tf.gather(w, l)), num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH)
    if aug_kind == 'shift':
        def shift(x, y, sw):
            x = random_resized_crop(x, img); x = colour_jitter(x); x = random_erasing(x, img)
            x, y, sw = cutmix(x, y, sw, img)
            return tf.clip_by_value(p1(x, training=True), 0, 255), y, sw
        ds = ds.map(shift, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda x, y, sw: (tf.clip_by_value(p1(x, training=True), 0, 255), y, sw), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)
def eval_ds(base, n_classes):
    return base.map(lambda x, l: (tf.cast(x, tf.float32), tf.one_hot(l, n_classes))).batch(BATCH).prefetch(tf.data.AUTOTUNE)

In [ ]:
# 4. Models (as Phase 1, with a variable input size) and the macro-F1 stopper
def build_model(arch, n_classes, img):
    inp = keras.Input((img, img, 3), name='image_0_255')
    if arch == 'MobileNetV2':
        x = keras.layers.Rescaling(1 / 127.5, offset=-1, name='mnv2_scale')(inp)
        base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(img, img, 3))
    else:
        x = inp; base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img, img, 3))
    base.trainable = False
    x = base(x, training=False); x = keras.layers.GlobalAveragePooling2D(name='gap')(x); x = keras.layers.Dropout(0.2, name='dropout')(x)
    return keras.Model(inp, keras.layers.Dense(n_classes, activation='softmax', name='probs')(x), name=f'{arch}_{n_classes}cls_{img}'), base
def unfreeze_final_blocks(base, arch):
    prefix = RECIPE['unfreeze_from'][arch]; names = [l.name for l in base.layers]
    idx = next((i for i, n in enumerate(names) if n.startswith(prefix)), int(len(names) * 0.75))
    base.trainable = True
    for l in base.layers[:idx]: l.trainable = False
    return {'first_trainable_layer': names[idx], 'trainable_backbone_layers': len(names) - idx, 'total_backbone_layers': len(names)}
class MacroF1Stopper(keras.callbacks.Callback):
    def __init__(self, sel_ds, y_sel, patience, log, phase):
        super().__init__(); self.sel_ds, self.y_sel, self.patience, self.log, self.phase = sel_ds, y_sel, patience, log, phase
        self.best, self.best_w, self.wait, self.t0 = -1.0, None, 0, None
    def on_epoch_begin(self, epoch, logs=None): self.t0 = time.time()
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.sel_ds, verbose=0).argmax(1); f1 = f1_score(self.y_sel, p, average='macro'); acc = accuracy_score(self.y_sel, p)
        self.log.append({'phase': self.phase, 'epoch': epoch + 1, 'train_loss': float(logs.get('loss', np.nan)), 'select_macro_f1': float(f1), 'select_accuracy': float(acc), 'seconds': round(time.time() - self.t0, 1)})
        print(f'  {self.phase} epoch {epoch+1}: loss {logs.get("loss", 0):.4f}  selection macro-F1 {f1:.4f}  acc {acc:.4f}  ({self.log[-1]["seconds"]}s)')
        if f1 > self.best + 1e-6: self.best, self.best_w, self.wait = f1, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience: self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None: self.model.set_weights(self.best_w)

In [ ]:
# ---- OCDF F4 calibration, abstention and selection helpers (Notebook 04) ----
import numpy as np

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(z, y, T=1.0):
    p = softmax(z, T)
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))))

def fit_temperature(z, y, lo=0.05, hi=20.0):
    """Calibration: positive T minimising negative log-likelihood on the calibration partition (Guo et al., 2017).
    Golden-section search on log T (NLL is unimodal in T for fixed logits)."""
    a, b = np.log(lo), np.log(hi); g = (np.sqrt(5) - 1) / 2
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = nll(z, y, np.exp(c)), nll(z, y, np.exp(d))
    for _ in range(80):
        if fc < fd: b, d, fd = d, c, fc; c = b - g * (b - a); fc = nll(z, y, np.exp(c))
        else: a, c, fc = c, d, fd; d = a + g * (b - a); fd = nll(z, y, np.exp(d))
    T = float(np.exp((a + b) / 2))
    return {'T': round(T, 4), 'nll_before': round(nll(z, y, 1.0), 4), 'nll_after': round(nll(z, y, T), 4)}

def brier(p, y):
    oh = np.zeros_like(p); oh[np.arange(len(y)), y] = 1
    return float(np.mean(np.sum((p - oh) ** 2, axis=1)))

def ece(p, y, bins=15):
    conf = p.max(1); pred = p.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= edges[0]) & (conf <= edges[1])
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def threshold_table(p, y, groups, thresholds=None):
    """Coverage, selective accuracy and accepted groups for each threshold (see docs/METHODS.md)."""
    thresholds = np.round(np.arange(0.50, 0.995, 0.01), 2) if thresholds is None else thresholds
    conf = p.max(1); pred = p.argmax(1); groups = np.asarray(groups); rows = []
    for t in thresholds:
        acc_mask = conf >= t; n_acc = int(acc_mask.sum())
        rows.append({'threshold': float(t), 'coverage': n_acc / len(y),
                     'selective_accuracy': float((pred[acc_mask] == y[acc_mask]).mean()) if n_acc else None,
                     'selective_risk': float((pred[acc_mask] != y[acc_mask]).mean()) if n_acc else None,
                     'correct_accepted_yield': float(((pred == y) & acc_mask).mean()),
                     'accepted_groups': int(len(set(groups[acc_mask]))), 'accepted': n_acc})
    return rows

def choose_threshold(rows, min_acc=0.95, min_cov=0.50, min_groups=50, fallback=0.90):
    """Highest-coverage threshold with selective accuracy >= 0.95, coverage >= 0.50 and >= 50 accepted groups;
    otherwise 'target infeasible' and the predeclared 0.90 for description only."""
    ok = [r for r in rows if r['selective_accuracy'] is not None and r['selective_accuracy'] >= min_acc
          and r['coverage'] >= min_cov and r['accepted_groups'] >= min_groups]
    if ok:
        best = max(ok, key=lambda r: (r['coverage'], -r['threshold']))
        return {'threshold': best['threshold'], 'status': 'target met', **{k: best[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}
    fb = min(rows, key=lambda r: abs(r['threshold'] - fallback))
    return {'threshold': fallback, 'status': 'target infeasible (0.90 used for description only)',
            **{k: fb[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}

def per_class_coverage(p, y, t, classes):
    conf = p.max(1); out = {}
    for i, c in enumerate(classes):
        m = y == i
        out[c] = round(float((conf[m] >= t).mean()), 4) if m.any() else None
    return out

def select_architecture(agg_rows, size_bytes=None, margin=0.01):
    """Pre-declared selection rule: highest mean selection macro-F1; within `margin` of the best, the smaller converted file."""
    best = max(agg_rows, key=lambda r: r['mean_macro_f1'])
    close = [r for r in agg_rows if best['mean_macro_f1'] - r['mean_macro_f1'] <= margin + 1e-12]
    if len(close) > 1 and size_bytes:
        pick = min(close, key=lambda r: size_bytes[r['architecture']])
        return pick['architecture'], f'within {margin} of best; smallest FP32 file chosen'
    return best['architecture'], (f"highest mean selection macro-F1 ({best['mean_macro_f1']:.4f}); "
                                   f"no other architecture within {margin}")
# ---- OCDF F6 evaluation helpers (Notebook 05): metrics, abstention, group bootstrap ----
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def full_metrics(p, y, eval_labels, n_out, threshold=None, groups=None):
    """Evaluation metrics (docs/METHODS.md). `eval_labels`: class indices present in this evaluation set (others are 'not evaluated').
    Predictions keep all n_out outputs (never masked), so predicting an absent class counts as an error."""
    pred = p.argmax(1); L = list(eval_labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=L, zero_division=0)
    out = {'n_images': int(len(y)), 'n_groups': int(len(set(groups))) if groups is not None else None,
           'accuracy': float(accuracy_score(y, pred)),
           'balanced_accuracy': float(np.mean([np.mean(pred[y == c] == c) for c in L])),
           'macro_f1': float(f1_score(y, pred, labels=L, average='macro', zero_division=0)),
           'per_class': {int(c): {'precision': float(a), 'recall': float(b), 'f1': float(d), 'support': int(s)}
                         for c, a, b, d, s in zip(L, pr, rc, f1, sup)},
           'confusion': confusion_matrix(y, pred, labels=list(range(n_out))).tolist(),
           'nll': float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))),
           'brier': float(np.mean(np.sum((p - np.eye(n_out)[y]) ** 2, axis=1))),
           'ece_15': ece15(p, y)}
    if threshold is not None:
        acc = p.max(1) >= threshold; n = int(acc.sum())
        out.update({'threshold': float(threshold), 'coverage': float(acc.mean()),
                    'selective_accuracy': float((pred[acc] == y[acc]).mean()) if n else None,
                    'selective_risk': float((pred[acc] != y[acc]).mean()) if n else None,
                    'correct_accepted_yield': float(((pred == y) & acc).mean()),
                    'coverage_per_class': {int(c): float(acc[y == c].mean()) for c in L}})
    return out

def ece15(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float); e = 0.0
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def fast_accuracy(y, pred):
    return float(np.mean(y == pred))

def fast_macro_f1(y, pred, labels, n_out):
    """Macro-F1 over `labels` from a bincount confusion matrix; identical to sklearn f1_score(average='macro', zero_division=0)."""
    cm = np.bincount(y * n_out + pred, minlength=n_out * n_out).reshape(n_out, n_out)
    tp = np.diag(cm)[labels].astype(float); fp = cm[:, labels].sum(0) - tp; fn = cm[labels, :].sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)))

def group_bootstrap(stat_fn, groups, n_boot=2000, seed=2026):
    """95% percentile interval of stat_fn(index_array), resampling whole groups with replacement (Efron & Tibshirani, 1994)."""
    groups = np.asarray(groups); uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind='stable'); lengths = np.bincount(inv); starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    rng = np.random.default_rng(seed); vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq)); lens = lengths[pick]
        offs = np.repeat(starts[pick] - np.cumsum(lens) + lens, lens) + np.arange(lens.sum())
        vals[b] = stat_fn(order[offs])
    return {'estimate': float(stat_fn(np.arange(len(groups)))), 'ci95': [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))],
            'n_boot': n_boot, 'n_groups': int(len(uniq))}

def reliability_bins(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float)
    idx = np.minimum((conf * bins).astype(int), bins - 1); rows = []
    for b in range(bins):
        m = idx == b
        rows.append({'bin_low': b / bins, 'bin_high': (b + 1) / bins, 'n': int(m.sum()),
                     'mean_confidence': float(conf[m].mean()) if m.any() else None, 'accuracy': float(correct[m].mean()) if m.any() else None})
    return rows

def risk_coverage(p, y):
    order = np.argsort(-p.max(1)); correct = (p.argmax(1) == y)[order]
    k = np.arange(1, len(y) + 1)
    return k / len(y), 1 - np.cumsum(correct) / k


# 4. E11b: shift-augmented counterparts of the Phase 1 models (Phase 1 data, recipe and label sets; seed 42), then T and threshold by the Phase 1 rules
def load_array_df(df):
    ds = tf.data.Dataset.from_tensor_slices([f'{WORK}/{p}' for p in df.working_path]).map(lambda p: load_img(p, IMG), num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    return np.concatenate([b.numpy() for b in ds])
def per_class_coverage(p, y, thr, classes): return {c: float((p.max(1)[y == i] >= thr).mean()) for i, c in enumerate(classes)}
def run_e11b(crop, arch, seed=42):
    rid = f'E11b_{crop}_{arch}_shift_s{seed}'; rdir = f'{RUNS}/{rid}'
    if os.path.exists(f'{rdir}/done.json'): print('done already:', rid); return json.load(open(f'{rdir}/done.json'))
    os.makedirs(rdir, exist_ok=True); classes = CLASSES[crop]
    tr = DEV[(DEV.crop == crop) & (DEV.partition == 'train')]; se = DEV[(DEV.crop == crop) & (DEV.partition == 'select')]
    cw = protocol2['class_weights_from_training_counts'][crop]; weights = [cw[c] for c in classes]
    keras.utils.set_random_seed(seed); keras.backend.clear_session()
    tds = train_ds(base_ds(tr, classes, IMG), len(classes), seed, weights, IMG, 'shift'); sel_ds = eval_ds(base_ds(se, classes, IMG), len(classes))
    y_sel = np.array([classes.index(l) for l in se.research_label])
    model, base = build_model(arch, len(classes), IMG); log = []; t0 = time.time()
    print(f'=== {rid}: train {len(tr)}, selection {len(se)} ===')
    model.compile(optimizer=keras.optimizers.Adam(RECIPE['head_lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(tds, epochs=RECIPE['head_max_epochs'], verbose=0, callbacks=[MacroF1Stopper(sel_ds, y_sel, RECIPE['patience'], log, 'head')])
    ufz = unfreeze_final_blocks(base, arch)
    model.compile(optimizer=keras.optimizers.Adam(RECIPE['finetune_lr']), loss='categorical_crossentropy', metrics=['accuracy'])
    st = MacroF1Stopper(sel_ds, y_sel, RECIPE['patience'], log, 'finetune'); st.best = max(r['select_macro_f1'] for r in log); st.best_w = model.get_weights()
    model.fit(tds, epochs=RECIPE['finetune_max_epochs'], verbose=0, callbacks=[st])
    model.save(f'{rdir}/best_model.keras'); pd.DataFrame(log).to_csv(f'{rdir}/history.csv', index=False)
    sel_p = model.predict(sel_ds, verbose=0)
    # calibration (Phase 1 rules): T on calib_T partition, threshold on thresh partition
    parts = {p: DEV[(DEV.crop == crop) & (DEV.partition == p)].sort_values('sample_id') for p in ['calib_T', 'thresh']}
    pc = model.predict(load_array_df(parts['calib_T']).astype(np.float32), batch_size=64, verbose=0); yc = np.array([classes.index(l) for l in parts['calib_T'].research_label])
    cal = fit_temperature(np.log(np.clip(pc, 1e-12, 1)), yc)
    pt = model.predict(load_array_df(parts['thresh']).astype(np.float32), batch_size=64, verbose=0); yt = np.array([classes.index(l) for l in parts['thresh'].research_label])
    ptT = softmax(np.log(np.clip(pt, 1e-12, 1)), cal['T']); curve = threshold_table(ptT, yt, parts['thresh'].group_id.values); ch = choose_threshold(curve)
    info = {'run_id': rid, 'experiment': 'E11b', 'crop': crop, 'architecture': arch, 'arm': 'shift', 'seed': seed, 'classes': classes, 'n_train': len(tr), 'n_select': len(se),
            'select_macro_f1': float(f1_score(y_sel, sel_p.argmax(1), average='macro')), 'select_accuracy': float(accuracy_score(y_sel, sel_p.argmax(1))),
            'unfreeze': ufz, 'epochs_run': {'head': sum(r['phase'] == 'head' for r in log), 'finetune': sum(r['phase'] == 'finetune' for r in log)},
            'temperature_T': cal['T'], 'calibration': cal, 'threshold': ch['threshold'], 'threshold_status': ch['status'],
            'thresh_coverage_per_class': per_class_coverage(ptT, yt, ch['threshold'], classes), 'minutes': round((time.time() - t0) / 60, 1),
            'environment': env, 'registration_sha256': reg_sha}
    json.dump(info, open(f'{rdir}/done.json', 'w'), indent=1, default=float)
    print(f'>>> {rid}: selection macro-F1 {info["select_macro_f1"]:.4f}  T {cal["T"]}  threshold {ch["threshold"]} ({ch["status"]})  [{info["minutes"]} min]')
    del model; gc.collect(); return info
# Phase 1 selection macro-F1 of the deployed models, for decision rule 1 (recomputed here from the frozen Keras weights)
P1 = {}
for crop in CROPS_TO_RUN:
    se = DEV[(DEV.crop == crop) & (DEV.partition == 'select')]; classes = CLASSES[crop]
    m = keras.models.load_model(f'{DRIVE}/runs/{crop}_EfficientNetB0_s42/best_model.keras')
    p = m.predict(eval_ds(base_ds(se, classes, IMG), len(classes)), verbose=0); y = np.array([classes.index(l) for l in se.research_label])
    P1[crop] = {'select_macro_f1': float(f1_score(y, p.argmax(1), average='macro')), 'select_accuracy': float(accuracy_score(y, p.argmax(1)))}
    print(crop, 'Phase 1 EfficientNet-B0 s42 selection macro-F1', round(P1[crop]['select_macro_f1'], 4)); del m
E11B = {}
for crop in CROPS_TO_RUN:
    for arch in ['EfficientNetB0', 'MobileNetV2']:
        E11B[(crop, arch)] = run_e11b(crop, arch)
json.dump({'phase1_selection': P1, 'e11b': {f'{c}_{a}': v for (c, a), v in E11B.items()}}, open(f'{OUT}/e11b_summary_{"_".join(CROPS_TO_RUN)}.json', 'w'), indent=1, default=float)

In [ ]:
# 5. Adaptation methods (label-free: AdaBN, Tent, gated self-training; labelled few-shot reference), evaluation on the test halves
def temperature(p, T):
    z = np.log(np.clip(p, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def predict_probs(model, X, bs=64):
    ds = tf.data.Dataset.from_tensor_slices(X).map(lambda x: tf.cast(x, tf.float32)).batch(bs).prefetch(tf.data.AUTOTUNE)
    return model.predict(ds, verbose=0)
def load_array(df):
    ds = tf.data.Dataset.from_tensor_slices([f'{WORK}/{p}' for p in df.working_path]).map(lambda p: load_img(p, IMG), num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    return np.concatenate([b.numpy() for b in ds])
def fresh(path):
    keras.backend.clear_session(); gc.collect(); return keras.models.load_model(path)
def get_base(model): return [l for l in model.layers if isinstance(l, keras.Model)][0]
def bn_layers(base): return [l for l in base.layers if isinstance(l, keras.layers.BatchNormalization)]

def adabn(model, X, bs=32):
    """Adaptive batch normalisation (Li et al., 2018): the backbone's BN moving statistics are replaced by statistics of the
    target images, as a running average over batches (momentum 1 - 1/k on the k-th batch). No weight is changed."""
    base = get_base(model); bns = bn_layers(base)
    for bn in bns: bn.trainable = True                      # Keras updates moving statistics only when the layer is trainable
    before = [np.array(bn.moving_mean) for bn in bns[:3]]
    bs = min(bs, len(X)); k = 0
    for s in range(0, len(X), bs):
        k += 1
        for bn in bns: bn.momentum = 1.0 - 1.0 / k
        base(tf.cast(X[s:s + bs], tf.float32), training=True)
    for bn in bns: bn.momentum = 0.99
    shift = float(np.mean([np.abs(np.array(bn.moving_mean) - b).mean() for bn, b in zip(bns[:3], before)]))
    assert shift > 0, 'AdaBN did not change the BN statistics (check trainable flags / training=True call)'
    return {'bn_layers': len(bns), 'batches': k, 'mean_abs_shift_first3_bn': shift}

def entropy_loss(y_true, y_pred):
    return -tf.reduce_sum(y_pred * tf.math.log(y_pred + 1e-8), axis=-1)

def tent(model, X, n_out, epochs=5, lr=1e-3, seed=0):
    """Tent (Wang et al., 2021): after re-estimating the BN statistics, the BN scale and shift parameters are trained to minimise
    the entropy of the model's predictions on the target images. Nothing else is trainable."""
    info = adabn(model, X)
    base = get_base(model)
    for l in model.layers: l.trainable = False
    base.trainable = True
    for l in base.layers: l.trainable = isinstance(l, keras.layers.BatchNormalization)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=entropy_loss)
    ds = (tf.data.Dataset.from_tensor_slices((X, np.zeros((len(X), n_out), np.float32))).shuffle(1024, seed=seed, reshuffle_each_iteration=True)
          .map(lambda x, y: (tf.cast(x, tf.float32), y)).batch(min(32, len(X))).prefetch(tf.data.AUTOTUNE))
    h = model.fit(ds, epochs=epochs, verbose=0)
    info.update({'epochs': epochs, 'lr': lr, 'entropy_first': float(h.history['loss'][0]), 'entropy_last': float(h.history['loss'][-1]),
                 'trainable_params': int(sum(np.prod(v.shape) for v in model.trainable_variables))})
    return info

def finetune(model, X, y, n_out, epochs, lr=1e-4, seed=0):
    """Phase 1 fine-tuning convention: head plus the final backbone blocks trainable, BN statistics fixed, Phase 1 augmentation."""
    base = get_base(model)
    for l in model.layers: l.trainable = False
    model.get_layer('probs').trainable = True
    ufz = unfreeze_final_blocks(base, 'EfficientNetB0')
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='categorical_crossentropy')
    ds = train_ds(tf.data.Dataset.from_tensor_slices((X, np.asarray(y, np.int64))), n_out, seed, np.ones(n_out, np.float32), IMG, 'phase1')
    h = model.fit(ds, epochs=epochs, verbose=0)
    return {'epochs': epochs, 'lr': lr, 'loss_first': float(h.history['loss'][0]), 'loss_last': float(h.history['loss'][-1]), 'first_trainable_layer': ufz['first_trainable_layer']}

def self_training(model, X, y_true, T, thr, n_out, rounds=2, epochs=10, seed=0):
    """Gated self-training: the frozen abstention rule (temperature T, threshold thr) decides which target images the model may
    label itself; accepted images and their pseudo-labels fine-tune the final blocks; two rounds. y_true is used only to report
    how accurate the pseudo-labels were (never for training)."""
    info = []
    for r in range(rounds):
        p = temperature(predict_probs(model, X), T); conf = p.max(1); acc = conf >= thr; pl = p.argmax(1)
        rec = {'round': r + 1, 'accepted': int(acc.sum()), 'of': int(len(X)),
               'pseudo_label_accuracy_on_accepted': float((pl[acc] == y_true[acc]).mean()) if acc.any() else None,
               'accepted_label_counts': {int(c): int(n) for c, n in zip(*np.unique(pl[acc], return_counts=True))}}
        if acc.sum() < 2:
            rec['action'] = 'stopped: fewer than two accepted images'; info.append(rec); break
        rec['finetune'] = finetune(model, X[acc], pl[acc], n_out, epochs=epochs, seed=seed + r); info.append(rec)
    return info

def evaluate(model, crop, T, thr):
    p = temperature(predict_probs(model, TESTX[crop]), T); y = TESTY[crop]; labels = EVAL_LABELS[crop]; n_out = N_OUT[crop]
    pred = p.argmax(1); conf = p.max(1); acc = conf >= thr
    return {'macro_f1': fast_macro_f1(y, pred, labels, n_out), 'accuracy': float((pred == y).mean()), 'coverage': float(acc.mean()),
            'selective_accuracy': float((pred[acc] == y[acc]).mean()) if acc.any() else None, 'mean_confidence': float(conf.mean()),
            'recall': {INV[crop][l]: float((pred[y == l] == l).mean()) for l in labels}}, pd.DataFrame({'y': y, 'pred': pred, 'conf': conf})

def draw_adapt(crop, N, draw):
    """N images drawn group-wise (whole capture groups, random order) from the crop's adapt half; the last group is cut to reach N."""
    d = ADAPT[crop]; rng = np.random.default_rng(2026 * 1000 + N * 10 + draw)
    gids = d.group_id.unique().copy(); rng.shuffle(gids); rows = []
    for g in gids:
        rows.append(d[d.group_id == g])
        if sum(len(r) for r in rows) >= N: break
    sel = pd.concat(rows).iloc[:N]
    return load_array(sel), np.array([LABEL_IDX[crop][l] for l in sel.research_label]), sel

def run_adaptation(crop, start, method, N, draw):
    rid = f'E13_{crop}_{start}_{method}_N{N}_d{draw}'; rpath = f'{RUNDIR}/{rid}.json'
    if os.path.exists(rpath): return json.load(open(rpath))
    t0 = time.time(); model = fresh(STARTS[crop][start]['path']); T, thr = STARTS[crop][start]['T'], STARTS[crop][start]['threshold']; n_out = N_OUT[crop]
    X, y, sel = draw_adapt(crop, N, draw)
    if method == 'adabn': detail = adabn(model, X)
    elif method == 'tent': detail = tent(model, X, n_out, seed=draw)
    elif method == 'self_training': detail = self_training(model, X, y, T, thr, n_out, seed=draw)
    elif method == 'labelled': detail = finetune(model, X, y, n_out, epochs=20, seed=draw)
    else: raise ValueError(method)
    ev, preds = evaluate(model, crop, T, thr); preds.to_csv(f'{RUNDIR}/{rid}_test_predictions.csv', index=False)
    rec = {'run_id': rid, 'crop': crop, 'start': start, 'method': method, 'N': N, 'draw': draw, 'n_adapt_groups': int(sel.group_id.nunique()),
           'adapt_label_counts': sel.research_label.value_counts().to_dict(), 'label_free': method != 'labelled', 'detail': detail,
           'test_set': TESTNAME[crop], **ev, 'minutes': round((time.time() - t0) / 60, 2)}
    json.dump(rec, open(rpath, 'w'), indent=1, default=float)
    print(f'  {rid}: macro-F1 {ev["macro_f1"]:.4f} (baseline {BASELINE[(crop, start)]["macro_f1"]:.4f}) cov {ev["coverage"]:.2f} [{rec["minutes"]} min]')
    del model; gc.collect(); return rec

In [ ]:
# 6. The E13 grid (resumable): 2 starting models x 4 methods x 6 N x 3 draws per crop, plus the unadapted baselines
STARTS, TESTX, TESTY, BASELINE = {}, {}, {}, {}
for crop in CROPS_TO_RUN:
    e = E11B[(crop, 'EfficientNetB0')]
    STARTS[crop] = {'phase1': {'path': f'{DRIVE}/runs/{crop}_EfficientNetB0_s42/best_model.keras', 'T': float(DEP[crop]['temperature_T']), 'threshold': float(DEP[crop]['threshold'])},
                    'shift':  {'path': f'{RUNS}/E11b_{crop}_EfficientNetB0_shift_s42/best_model.keras', 'T': float(e['temperature_T']), 'threshold': float(e['threshold'])}}
    TESTX[crop] = load_array(TESTDF[crop]); TESTY[crop] = np.array([LABEL_IDX[crop][l] for l in TESTDF[crop].research_label])
    print(crop, 'test half loaded', TESTX[crop].shape)
    for start in ['phase1', 'shift']:
        bpath = f'{RUNDIR}/E13_{crop}_{start}_baseline.json'
        if os.path.exists(bpath): BASELINE[(crop, start)] = json.load(open(bpath))
        else:
            m = fresh(STARTS[crop][start]['path']); ev, preds = evaluate(m, crop, STARTS[crop][start]['T'], STARTS[crop][start]['threshold'])
            preds.to_csv(f'{RUNDIR}/E13_{crop}_{start}_baseline_test_predictions.csv', index=False)
            BASELINE[(crop, start)] = {'crop': crop, 'start': start, 'method': 'none', 'N': 0, 'draw': 0, 'label_free': True, 'test_set': TESTNAME[crop], **ev}
            json.dump(BASELINE[(crop, start)], open(bpath, 'w'), indent=1, default=float); del m
        print(f'  baseline {start}: macro-F1 {BASELINE[(crop, start)]["macro_f1"]:.4f} acc {BASELINE[(crop, start)]["accuracy"]:.4f} cov {BASELINE[(crop, start)]["coverage"]:.2f}')
t_all = time.time(); n_done = 0
for crop in CROPS_TO_RUN:
    for start in ['phase1', 'shift']:
        for N in NS:
            for draw in DRAWS:
                for method in METHODS:
                    run_adaptation(crop, start, method, N, draw); n_done += 1
            print(f'--- {crop} {start} N={N} finished ({(time.time() - t_all) / 60:.0f} min elapsed)')
print('finished requested runs:', n_done)

In [ ]:
# 7. Curves, decision rule (v3 adaptation registered), source probe, figure, zip   (run after BOTH crops have finished)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
rows = [json.load(open(p)) for p in sorted(glob.glob(f'{RUNDIR}/E13_*.json'))]
R = pd.DataFrame([{k: v for k, v in r.items() if k not in ('detail', 'recall', 'adapt_label_counts')} | {f'recall_{k}': v for k, v in r.get('recall', {}).items()} for r in rows])
R.to_csv(f'{OUT}/e13_runs.csv', index=False)
C = (R[R.method != 'none'].groupby(['crop', 'start', 'method', 'N']).agg(macro_f1_mean=('macro_f1', 'mean'), macro_f1_sd=('macro_f1', 'std'), accuracy_mean=('accuracy', 'mean'),
     coverage_mean=('coverage', 'mean'), draws=('draw', 'count')).reset_index())
base = R[R.method == 'none'].set_index(['crop', 'start']).macro_f1
C['baseline_macro_f1'] = [base.get((c, s), np.nan) for c, s in zip(C.crop, C.start)]; C['gain'] = C.macro_f1_mean - C.baseline_macro_f1
C.to_csv(f'{OUT}/e13_curves.csv', index=False)
pd.set_option('display.width', 220)
for crop in sorted(C.crop.unique()):
    print(f'\n=== {crop}: mean macro-F1 on {TESTNAME[crop]} over 3 draws (baseline: ' + ', '.join(f'{s} {base.get((crop, s), np.nan):.4f}' for s in ['phase1', 'shift']) + ') ===')
    print(C[C.crop == crop].pivot_table(index=['start', 'method'], columns='N', values='macro_f1_mean').round(4).to_string())
# Decision rule (registered): label-free method and N with the highest mean macro-F1 on d3m_test, Phase 1 maize model
dec_path = f'{DRIVE}/records/v3_adaptation_registered.json'
if (C.crop == 'maize').any() and not os.path.exists(dec_path):
    cand = C[(C.crop == 'maize') & (C.start == 'phase1') & (C.method != 'labelled') & (C.draws == 3)]
    best = cand.sort_values(['macro_f1_mean', 'N'], ascending=[False, True]).iloc[0]
    cass = C[(C.crop == 'cassava') & (C.start == 'phase1') & (C.method != 'labelled') & (C.draws == 3)]
    cbest = cass.sort_values(['macro_f1_mean', 'N'], ascending=[False, True]).iloc[0] if len(cass) else None
    dec = {'registered_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'rule': 'label-free method and N with the highest mean macro-F1 on d3m_test over three draws, Phase 1 maize model',
           'maize': {'method': best.method, 'N': int(best.N), 'mean_macro_f1_d3m_test': float(best.macro_f1_mean), 'baseline_macro_f1_d3m_test': float(best.baseline_macro_f1),
                     'gain_on_d3m_test': float(best.gain)},
           'cassava_exploratory': None if cbest is None else {'method': cbest.method, 'N': int(cbest.N), 'mean_macro_f1_d3c_test': float(cbest.macro_f1_mean), 'baseline_macro_f1_d3c_test': float(cbest.baseline_macro_f1)},
           'application': 'Notebook 11: the Phase 1 maize model is adapted once with this method and N on images drawn from d5_adapt with the draw-1 seed rule of Notebook 09, '
                          'then scored on the sealed d5_test; draws 2 and 3 are reported as sensitivity. Primary endpoint: paired group-bootstrap difference in macro-F1 '
                          '(M01, M05) versus the unadapted Phase 1 model; confirmed if the 95% interval excludes zero and the estimate is >= +0.02.',
           'method_settings': {'adabn': 'running-average BN statistics over batches of 32', 'tent': 'AdaBN then 5 epochs, Adam 1e-3, BN scale/shift only',
                               'self_training': 'two rounds; accept if max temperature-scaled probability >= Phase 1 threshold; 10 epochs Adam 1e-4 on final blocks + head, Phase 1 augmentation'},
           'registration_sha256': reg_sha, 'e13_runs_sha256': hashlib.sha256(open(f'{OUT}/e13_runs.csv', 'rb').read()).hexdigest()}
    json.dump(dec, open(dec_path, 'w'), indent=1); open(dec_path + '.sha256', 'w').write(hashlib.sha256(open(dec_path, 'rb').read()).hexdigest())
    print('\nREGISTERED v3 adaptation:', json.dumps(dec['maize']), '| sha', open(dec_path + '.sha256').read()[:16])
elif os.path.exists(dec_path): print('\nv3 adaptation already registered:', json.load(open(dec_path))['maize'])
else: print('\n(maize runs not present in this session: the decision is written when cell 7 runs with maize results on Drive)')
# Figure: adaptation curves
crops = sorted(C.crop.unique()); fig, axes = plt.subplots(len(crops), 2, figsize=(12, 4.2 * len(crops)), squeeze=False)
for i, crop in enumerate(crops):
    for j, start in enumerate(['phase1', 'shift']):
        ax = axes[i, j]; d = C[(C.crop == crop) & (C.start == start)]
        for m in METHODS:
            dm = d[d.method == m].sort_values('N')
            if len(dm): ax.errorbar(dm.N, dm.macro_f1_mean, yerr=dm.macro_f1_sd.fillna(0), marker='o', capsize=3, label=m, linestyle='--' if m == 'labelled' else '-')
        if (crop, start) in base.index: ax.axhline(base[(crop, start)], color='grey', linestyle=':', label='unadapted')
        ax.set_xscale('log'); ax.set_xticks(NS); ax.set_xticklabels(NS); ax.set_xlabel('N target images (unlabelled; labelled for the reference)'); ax.set_ylabel(f'macro-F1 on {TESTNAME[crop]}')
        ax.set_title(f'{crop}: start = {"Phase 1 model" if start == "phase1" else "shift-augmented counterpart"}'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(f'{OUT}/figure_e13_adaptation_curves.png', dpi=130); plt.show(); plt.close(fig)
# Source probe (cassava, secondary): can a linear probe on frozen GAP features tell Uganda (D1 test) from Ghana (d3c_test), before and after the best label-free adaptation?
probe = None
if 'cassava' in CROPS_TO_RUN and (C.crop == 'cassava').any():
    from sklearn.linear_model import LogisticRegression; from sklearn.model_selection import cross_val_score
    ug = M[(M.source_id == 'D1') & (M.partition == 'test')].sort_values('sample_id'); Xug = load_array(ug)
    def probe_auc(model):
        feat = keras.Model(model.inputs, model.get_layer('gap').output)
        f = np.concatenate([feat.predict(Xug.astype(np.float32), batch_size=64, verbose=0), feat.predict(TESTX['cassava'].astype(np.float32), batch_size=64, verbose=0)])
        ysrc = np.r_[np.zeros(len(Xug)), np.ones(len(TESTX['cassava']))]
        return float(cross_val_score(LogisticRegression(max_iter=2000), f, ysrc, cv=5, scoring='roc_auc').mean())
    cb = C[(C.crop == 'cassava') & (C.start == 'phase1') & (C.method != 'labelled') & (C.draws == 3)].sort_values(['macro_f1_mean', 'N'], ascending=[False, True]).iloc[0]
    m0 = fresh(STARTS['cassava']['phase1']['path']); auc_before = probe_auc(m0)
    X, y, _ = draw_adapt('cassava', int(cb.N), 1)
    if cb.method == 'adabn': adabn(m0, X)
    elif cb.method == 'tent': tent(m0, X, N_OUT['cassava'], seed=1)
    else: self_training(m0, X, y, STARTS['cassava']['phase1']['T'], STARTS['cassava']['phase1']['threshold'], N_OUT['cassava'], seed=1)
    auc_after = probe_auc(m0)
    probe = {'features': 'GAP features of the Phase 1 cassava model', 'sets': 'D1 test (Uganda) vs d3c_test (Ghana)', 'probe': 'logistic regression, 5-fold AUC',
             'auc_before': auc_before, 'auc_after': auc_after, 'adaptation': {'method': cb.method, 'N': int(cb.N), 'draw': 1}}
    json.dump(probe, open(f'{OUT}/e13_source_probe_cassava.json', 'w'), indent=1); print('\nsource probe (cassava):', json.dumps(probe))
import zipfile
zp = '/content/nb09_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{OUT}/*'):
        if os.path.isfile(p): z.write(p, os.path.basename(p))
    for p in glob.glob(f'{RUNDIR}/*.json'): z.write(p, f'e13_runs/{os.path.basename(p)}')
    for p in glob.glob(f'{RUNS}/E11b_*/done.json') + glob.glob(f'{RUNS}/E11b_*/history.csv'): z.write(p, f'runs_v3/{os.path.basename(os.path.dirname(p))}/{os.path.basename(p)}')
    for p in [dec_path, dec_path + '.sha256']:
        if os.path.exists(p): z.write(p, os.path.basename(p))
from google.colab import files
files.download(zp); print('nb09_results.zip ready')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb09_results.zip`. The copies used for the published results are in this repository under `records/nb09/`.

Run cell 7 once both crop sessions are done; it reads every finished run from Drive.